## SRP065487

**paper:** [PMID: 27212400](https://pmc.ncbi.nlm.nih.gov/articles/PMC5125026/) - Genetic Basis for Red Coloration in Birds, 2016

**date, curator:** 2026-03-26, Sara Carsanaro

**notes**
* remove DNA seq and qPCR libraries
* supplemental methods has majority of details about sample metadata

### annotation summary

In [23]:
anat_summary = library_to_add[['infoOrgan', 'anatId', 'anatName', 'anatAnnotationStatus']]
unique_anat = anat_summary.drop_duplicates()
display_df(unique_anat)

,infoOrgan,anatId,anatName,anatAnnotationStatus
0,Liver,UBERON:0002107,liver,perfect match
1,Skin,UBERON:0000014,zone of skin,perfect match


In [24]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

,infoStage,stageId,stageName,stageAnnotationStatus
0,Adult,UBERON:0000113,post-juvenile adult stage,perfect match


### set variables, import packages, define functions

In [1]:
experiment_id = "SRP065487"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

## validation
path_to_v_script = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/validate_annotations.py'
path_to_rules = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/rules/'
val_output = "{}{}/validation.tsv".format(path_to_output_main, experiment_id)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

### script

In [3]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
Be patient, it may take a few minutes.
100%|███████████████████████████████████████████| 12/12 [00:22<00:00,  1.88s/it]
0 samples dont have attributes, try to find them somewhere else
0it [00:00, ?it/s]
0 samples dont have attributes


### library annnotations

In [4]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX1430201,SRP065487,Illumina HiSeq 1500,SRS1161002,,,,,,Liver,Adult,,,,pooled male and female,Yellow,,9135,,,,,,YEL_LIV,SAMN04263680,,,,,,,,,,,26/03/2026,RNA-Seq of Yellow Canaries Liver,,YEL_LIV,,,,,,TRANSCRIPTOMIC,unspecified
1,SRX1430200,SRP065487,Illumina HiSeq 1500,SRS1161001,,,,,,Skin,Adult,,,,pooled male and female,Yellow,,9135,,,,,,YEL_SKIN,SAMN04260514,,,,,,,,,,,26/03/2026,RNA-Seq of Yellow Canaries skin,,YEL_SKIN,,,,,,TRANSCRIPTOMIC,unspecified
2,SRX1430090,SRP065487,Illumina HiSeq 1500,SRS1160891,,,,,,Liver,Adult,,,,pooled male and female,Red,,9135,,,,,,RED_LIV,SAMN04263679,,,,,,,,,,,26/03/2026,RNA-Seq of Red Canaries skin,,RED_LIV,,,,,,TRANSCRIPTOMIC,unspecified
3,SRX1430049,SRP065487,Illumina HiSeq 1500,SRS1160851,,,,,,Skin,Adult,,,,pooled male and female,Red,,9135,,,,,,RED_SKIN,SAMN04260516,,,,,,,,,,,26/03/2026,RNA-Seq of Red Canaries skin,,RED_SKIN,,,,,,TRANSCRIPTOMIC,unspecified


#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [5]:
unique_sorted(library, "infoOrgan")

['Liver' 'Skin']


In [6]:

# Liver
library.loc[library["infoOrgan"] == "Liver", "anatId"] = "UBERON:0002107"
library.loc[library["infoOrgan"] == "Liver", "anatName"] = "liver"
# perfect match, missing child term, other
library.loc[library["infoOrgan"] == "Liver", "anatAnnotationStatus"] = "perfect match"
# full sampling, partial sampling, not documented
library.loc[library["infoOrgan"] == "Liver", "anatBiologicalStatus"] = "not documented"

# Skin
library.loc[library["infoOrgan"] == "Skin", "anatId"] = "UBERON:0000014"
library.loc[library["infoOrgan"] == "Skin", "anatName"] = "zone of skin"
# perfect match, missing child term, other
library.loc[library["infoOrgan"] == "Skin", "anatAnnotationStatus"] = "perfect match"
# full sampling, partial sampling, not documented
library.loc[library["infoOrgan"] == "Skin", "anatBiologicalStatus"] = "not documented"

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX1430201,SRP065487,Illumina HiSeq 1500,SRS1161002,UBERON:0002107,liver,,,,Liver,Adult,perfect match,not documented,,pooled male and female,Yellow,,9135,,,,,,YEL_LIV,SAMN04263680,,,,,,,,,,,26/03/2026,RNA-Seq of Yellow Canaries Liver,,YEL_LIV,,,,,,TRANSCRIPTOMIC,unspecified
1,SRX1430200,SRP065487,Illumina HiSeq 1500,SRS1161001,UBERON:0000014,zone of skin,,,,Skin,Adult,perfect match,not documented,,pooled male and female,Yellow,,9135,,,,,,YEL_SKIN,SAMN04260514,,,,,,,,,,,26/03/2026,RNA-Seq of Yellow Canaries skin,,YEL_SKIN,,,,,,TRANSCRIPTOMIC,unspecified
2,SRX1430090,SRP065487,Illumina HiSeq 1500,SRS1160891,UBERON:0002107,liver,,,,Liver,Adult,perfect match,not documented,,pooled male and female,Red,,9135,,,,,,RED_LIV,SAMN04263679,,,,,,,,,,,26/03/2026,RNA-Seq of Red Canaries skin,,RED_LIV,,,,,,TRANSCRIPTOMIC,unspecified
3,SRX1430049,SRP065487,Illumina HiSeq 1500,SRS1160851,UBERON:0000014,zone of skin,,,,Skin,Adult,perfect match,not documented,,pooled male and female,Red,,9135,,,,,,RED_SKIN,SAMN04260516,,,,,,,,,,,26/03/2026,RNA-Seq of Red Canaries skin,,RED_SKIN,,,,,,TRANSCRIPTOMIC,unspecified


#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src/ontology/components)

In [7]:
unique_sorted(library, "infoStage")

['Adult']


In [8]:
# all
library.loc[:,'stageId'] = 'UBERON:0000113'
library.loc[:,'stageName'] = 'post-juvenile adult stage'
# perfect match, missing child term, other
library.loc[:,'stageAnnotationStatus'] = 'perfect match'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX1430201,SRP065487,Illumina HiSeq 1500,SRS1161002,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,Liver,Adult,perfect match,not documented,perfect match,pooled male and female,Yellow,,9135,,,,,,YEL_LIV,SAMN04263680,,,,,,,,,,,26/03/2026,RNA-Seq of Yellow Canaries Liver,,YEL_LIV,,,,,,TRANSCRIPTOMIC,unspecified
1,SRX1430200,SRP065487,Illumina HiSeq 1500,SRS1161001,UBERON:0000014,zone of skin,UBERON:0000113,post-juvenile adult stage,,Skin,Adult,perfect match,not documented,perfect match,pooled male and female,Yellow,,9135,,,,,,YEL_SKIN,SAMN04260514,,,,,,,,,,,26/03/2026,RNA-Seq of Yellow Canaries skin,,YEL_SKIN,,,,,,TRANSCRIPTOMIC,unspecified
2,SRX1430090,SRP065487,Illumina HiSeq 1500,SRS1160891,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,Liver,Adult,perfect match,not documented,perfect match,pooled male and female,Red,,9135,,,,,,RED_LIV,SAMN04263679,,,,,,,,,,,26/03/2026,RNA-Seq of Red Canaries skin,,RED_LIV,,,,,,TRANSCRIPTOMIC,unspecified
3,SRX1430049,SRP065487,Illumina HiSeq 1500,SRS1160851,UBERON:0000014,zone of skin,UBERON:0000113,post-juvenile adult stage,,Skin,Adult,perfect match,not documented,perfect match,pooled male and female,Red,,9135,,,,,,RED_SKIN,SAMN04260516,,,,,,,,,,,26/03/2026,RNA-Seq of Red Canaries skin,,RED_SKIN,,,,,,TRANSCRIPTOMIC,unspecified


#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

In [9]:
library.loc[:,'sex'] = 'mixed'
#library.loc[library["sex"] == "male", "sex"] = "M"
#library.loc[library["sex"] == "female", "sex"] = "F"

#library.loc[:,'strain'] = ''

#library.loc[:,'genotype'] = ''

#library.loc[:,'speciesId'] = ''

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX1430201,SRP065487,Illumina HiSeq 1500,SRS1161002,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,Liver,Adult,perfect match,not documented,perfect match,mixed,Yellow,,9135,,,,,,YEL_LIV,SAMN04263680,,,,,,,,,,,26/03/2026,RNA-Seq of Yellow Canaries Liver,,YEL_LIV,,,,,,TRANSCRIPTOMIC,unspecified
1,SRX1430200,SRP065487,Illumina HiSeq 1500,SRS1161001,UBERON:0000014,zone of skin,UBERON:0000113,post-juvenile adult stage,,Skin,Adult,perfect match,not documented,perfect match,mixed,Yellow,,9135,,,,,,YEL_SKIN,SAMN04260514,,,,,,,,,,,26/03/2026,RNA-Seq of Yellow Canaries skin,,YEL_SKIN,,,,,,TRANSCRIPTOMIC,unspecified
2,SRX1430090,SRP065487,Illumina HiSeq 1500,SRS1160891,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,Liver,Adult,perfect match,not documented,perfect match,mixed,Red,,9135,,,,,,RED_LIV,SAMN04263679,,,,,,,,,,,26/03/2026,RNA-Seq of Red Canaries skin,,RED_LIV,,,,,,TRANSCRIPTOMIC,unspecified
3,SRX1430049,SRP065487,Illumina HiSeq 1500,SRS1160851,UBERON:0000014,zone of skin,UBERON:0000113,post-juvenile adult stage,,Skin,Adult,perfect match,not documented,perfect match,mixed,Red,,9135,,,,,,RED_SKIN,SAMN04260516,,,,,,,,,,,26/03/2026,RNA-Seq of Red Canaries skin,,RED_SKIN,,,,,,TRANSCRIPTOMIC,unspecified


#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

In [10]:
# making these variables because we use them again in the experiment file
my_protocol = 'TruSeq RNA Sample Preparation Kit'
# full_length or 3'
my_protocolType = 'full_length'

library.loc[:,'protocol'] = my_protocol
library.loc[:,'protocolType'] = my_protocolType
# polyA, ribo-minus, miRNA, lncRNA, circRNA
library.loc[:,'RNASelection'] = 'polyA'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX1430201,SRP065487,Illumina HiSeq 1500,SRS1161002,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,Liver,Adult,perfect match,not documented,perfect match,mixed,Yellow,,9135,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,YEL_LIV,SAMN04263680,,,,,,,,,,,26/03/2026,RNA-Seq of Yellow Canaries Liver,,YEL_LIV,,,,,,TRANSCRIPTOMIC,unspecified
1,SRX1430200,SRP065487,Illumina HiSeq 1500,SRS1161001,UBERON:0000014,zone of skin,UBERON:0000113,post-juvenile adult stage,,Skin,Adult,perfect match,not documented,perfect match,mixed,Yellow,,9135,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,YEL_SKIN,SAMN04260514,,,,,,,,,,,26/03/2026,RNA-Seq of Yellow Canaries skin,,YEL_SKIN,,,,,,TRANSCRIPTOMIC,unspecified
2,SRX1430090,SRP065487,Illumina HiSeq 1500,SRS1160891,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,Liver,Adult,perfect match,not documented,perfect match,mixed,Red,,9135,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,RED_LIV,SAMN04263679,,,,,,,,,,,26/03/2026,RNA-Seq of Red Canaries skin,,RED_LIV,,,,,,TRANSCRIPTOMIC,unspecified
3,SRX1430049,SRP065487,Illumina HiSeq 1500,SRS1160851,UBERON:0000014,zone of skin,UBERON:0000113,post-juvenile adult stage,,Skin,Adult,perfect match,not documented,perfect match,mixed,Red,,9135,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,RED_SKIN,SAMN04260516,,,,,,,,,,,26/03/2026,RNA-Seq of Red Canaries skin,,RED_SKIN,,,,,,TRANSCRIPTOMIC,unspecified


#### globin, replicates

In [11]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

no duplicate values in SRSId


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# replicates
#library.loc[library["#libraryId"] == "old", "replicate"] = "1"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

#### sample age, pato, physiological status
* [PATO](https://www.ebi.ac.uk/ols4/ontologies/pato)
* [EFO](https://www.ebi.ac.uk/ols4/ontologies/efo)

In [ ]:
#library.loc[:,'sampleAge_value'] = ''
#library.loc[:,'sampleAge_unit'] = ''

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
#library.loc[:,'physiologicalStatus'] = ''

# ex. left, right
#library.loc[:,'EFOid'] = ''
#library.loc[:,'EFOname'] = ''

# view
display_df(library)

#### condition

In [ ]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
#library.loc[library["condition"] == "old", "condition"] = "new"

# view
display_df(library)

#### annotator id, last modification date

In [12]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2026-03-26'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX1430201,SRP065487,Illumina HiSeq 1500,SRS1161002,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,Liver,Adult,perfect match,not documented,perfect match,mixed,Yellow,,9135,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,YEL_LIV,SAMN04263680,,,,,,,,,,SAC,2026-03-26,RNA-Seq of Yellow Canaries Liver,,YEL_LIV,,,,,,TRANSCRIPTOMIC,unspecified
1,SRX1430200,SRP065487,Illumina HiSeq 1500,SRS1161001,UBERON:0000014,zone of skin,UBERON:0000113,post-juvenile adult stage,,Skin,Adult,perfect match,not documented,perfect match,mixed,Yellow,,9135,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,YEL_SKIN,SAMN04260514,,,,,,,,,,SAC,2026-03-26,RNA-Seq of Yellow Canaries skin,,YEL_SKIN,,,,,,TRANSCRIPTOMIC,unspecified
2,SRX1430090,SRP065487,Illumina HiSeq 1500,SRS1160891,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,Liver,Adult,perfect match,not documented,perfect match,mixed,Red,,9135,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,RED_LIV,SAMN04263679,,,,,,,,,,SAC,2026-03-26,RNA-Seq of Red Canaries skin,,RED_LIV,,,,,,TRANSCRIPTOMIC,unspecified
3,SRX1430049,SRP065487,Illumina HiSeq 1500,SRS1160851,UBERON:0000014,zone of skin,UBERON:0000113,post-juvenile adult stage,,Skin,Adult,perfect match,not documented,perfect match,mixed,Red,,9135,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,RED_SKIN,SAMN04260516,,,,,,,,,,SAC,2026-03-26,RNA-Seq of Red Canaries skin,,RED_SKIN,,,,,,TRANSCRIPTOMIC,unspecified


#### comments

In [13]:
library.loc[:,'comment'] = 'PMID: 27212400'

#### save complete file with correct columns

In [14]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX1430201,SRP065487,Illumina HiSeq 1500,SRS1161002,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,Liver,Adult,perfect match,not documented,perfect match,mixed,Yellow,,9135,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,YEL_LIV,SAMN04263680,,,,,,,PMID: 27212400,,,SAC,2026-03-26
1,SRX1430200,SRP065487,Illumina HiSeq 1500,SRS1161001,UBERON:0000014,zone of skin,UBERON:0000113,post-juvenile adult stage,,Skin,Adult,perfect match,not documented,perfect match,mixed,Yellow,,9135,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,YEL_SKIN,SAMN04260514,,,,,,,PMID: 27212400,,,SAC,2026-03-26
2,SRX1430090,SRP065487,Illumina HiSeq 1500,SRS1160891,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,Liver,Adult,perfect match,not documented,perfect match,mixed,Red,,9135,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,RED_LIV,SAMN04263679,,,,,,,PMID: 27212400,,,SAC,2026-03-26
3,SRX1430049,SRP065487,Illumina HiSeq 1500,SRS1160851,UBERON:0000014,zone of skin,UBERON:0000113,post-juvenile adult stage,,Skin,Adult,perfect match,not documented,perfect match,mixed,Red,,9135,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,RED_SKIN,SAMN04260516,,,,,,,PMID: 27212400,,,SAC,2026-03-26


### experiment annotations

In [15]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP065487,Red coloration in Birds,"Comparison of polymorphism in red siskins (Spinus cucullata), common canaries (Serinus canaria), and 30 red factor canaries (hybrid variety produced by crossing red siskins with yellow canaries), to infer regions of the genome responsible for red coloration in birds.",SRA,,,,,,,PRJNA300534,27212400,,10.1016/j.cub.2016.03.076,,


#### experiment and protocol details

In [16]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

4

In [17]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'partial'
# Bgee 1K
experiment.loc[:,'projectTags'] = 'Bgee 1K' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
experiment.loc[:,'protocol'] = my_protocol
experiment.loc[:,'protocolType'] = my_protocolType

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP065487,Red coloration in Birds,"Comparison of polymorphism in red siskins (Spinus cucullata), common canaries (Serinus canaria), and 30 red factor canaries (hybrid variety produced by crossing red siskins with yellow canaries), to infer regions of the genome responsible for red coloration in birds.",SRA,partial,Bgee 1K,4,TruSeq RNA Sample Preparation Kit,full_length,,PRJNA300534,27212400,,10.1016/j.cub.2016.03.076,,


#### paper and xrefs

In [18]:
#experiment.loc[:,'GSE'] = ''
#experiment.loc[:,'Bioproject'] = '' 
experiment.loc[:,'PMID'] = '27212400'
experiment.loc[:,'reference_url'] = 'https://pmc.ncbi.nlm.nih.gov/articles/PMC5125026/'
experiment.loc[:,'DOI'] = '10.1016/j.cub.2016.03.076'
#experiment.loc[:,'xrefs'] = ''

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP065487,Red coloration in Birds,"Comparison of polymorphism in red siskins (Spinus cucullata), common canaries (Serinus canaria), and 30 red factor canaries (hybrid variety produced by crossing red siskins with yellow canaries), to infer regions of the genome responsible for red coloration in birds.",SRA,partial,Bgee 1K,4,TruSeq RNA Sample Preparation Kit,full_length,,PRJNA300534,27212400,https://pmc.ncbi.nlm.nih.gov/articles/PMC5125026/,10.1016/j.cub.2016.03.076,,


#### comments

In [19]:
experiment.loc[:,'comment'] = 'removed WGS and qPCR libraries'

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP065487,Red coloration in Birds,"Comparison of polymorphism in red siskins (Spinus cucullata), common canaries (Serinus canaria), and 30 red factor canaries (hybrid variety produced by crossing red siskins with yellow canaries), to infer regions of the genome responsible for red coloration in birds.",SRA,partial,Bgee 1K,4,TruSeq RNA Sample Preparation Kit,full_length,,PRJNA300534,27212400,https://pmc.ncbi.nlm.nih.gov/articles/PMC5125026/,10.1016/j.cub.2016.03.076,,removed WGS and qPCR libraries


#### save complete file

In [20]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [21]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

In [22]:
! python3 $path_to_v_script --bulk-exp $experiment_to_add_path --bulk-lib $library_to_add_path --rules-dir $path_to_rules --out $val_output --strict

Total issues: 0
Errors: 0
Warnings: 0
Top codes:


#### check columns match

In [25]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

Already up to date.
The columns in the library file match
The columns in the experiment file match


#### view files

In [26]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
56877,SRX16111183,SRP385744,Illumina NovaSeq 6000,SRS13774216,UBERON:0002385,muscle tissue,UBERON:0000112,sexually immature stage,,Muscle,Juvenile - chick,perfect match,not documented,other,F,,,9089,TruSeq RNA Library Prep Kit,full_length,polyA,,,Muscle of a female golden pheasant,SAMN29625220,,,,,,,PMID: 39846188,,,SAC,2026-03-26
56878,ERX297947,ERP003755,Illumina HiSeq 2000,ERS337042,UBERON:0000955,brain,UBERON:0018241,prime adult stage,,brain,adult,perfect match,not documented,perfect match,M,,,9135,,,polyA,,,Serinus canaria brain tissues,SAMEA2168438,,,,,,,PMID: 25631560,,,SAC,2026-03-26
56879,SRX1430201,SRP065487,Illumina HiSeq 1500,SRS1161002,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,Liver,Adult,perfect match,not documented,perfect match,mixed,Yellow,,9135,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,YEL_LIV,SAMN04263680,,,,,,,PMID: 27212400,,,SAC,2026-03-26
56880,SRX1430200,SRP065487,Illumina HiSeq 1500,SRS1161001,UBERON:0000014,zone of skin,UBERON:0000113,post-juvenile adult stage,,Skin,Adult,perfect match,not documented,perfect match,mixed,Yellow,,9135,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,YEL_SKIN,SAMN04260514,,,,,,,PMID: 27212400,,,SAC,2026-03-26
56881,SRX1430090,SRP065487,Illumina HiSeq 1500,SRS1160891,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,Liver,Adult,perfect match,not documented,perfect match,mixed,Red,,9135,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,RED_LIV,SAMN04263679,,,,,,,PMID: 27212400,,,SAC,2026-03-26
56882,SRX1430049,SRP065487,Illumina HiSeq 1500,SRS1160851,UBERON:0000014,zone of skin,UBERON:0000113,post-juvenile adult stage,,Skin,Adult,perfect match,not documented,perfect match,mixed,Red,,9135,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,RED_SKIN,SAMN04260516,,,,,,,PMID: 27212400,,,SAC,2026-03-26


In [27]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
1125,SRP385744,Chrysolophus pictus diploid genome sequencing ...,We sequenced a female golden pheasant with Nan...,SRA,partial,Bgee 1K,7,TruSeq RNA Library Prep Kit,full_length,,PRJNA857339,39846188,https://pmc.ncbi.nlm.nih.gov/articles/PMC11891...,10.24272/j.issn.2095-8137.2024.237,,removed DNA libraries
1126,ERP003755,Transcriptome sequencing of Serinus Canaria an...,Transcriptome sequencing of Serinus Canaria an...,SRA,total,Bgee 1K,1,,,,PRJEB4463,25631560,https://pmc.ncbi.nlm.nih.gov/articles/PMC4373106/,10.1186/s13059-014-0578-9,,
1127,SRP065487,Red coloration in Birds,Comparison of polymorphism in red siskins (Spi...,SRA,partial,Bgee 1K,4,TruSeq RNA Sample Preparation Kit,full_length,,PRJNA300534,27212400,https://pmc.ncbi.nlm.nih.gov/articles/PMC5125026/,10.1016/j.cub.2016.03.076,,removed WGS and qPCR libraries


### add annotations to git

In [28]:
! git pull

Already up to date.


In [29]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [30]:
! git status

On branch develop
Your branch is up to date with 'origin/develop'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   ../../../RNA_Seq/RNASeqExperiment.tsv
	modified:   ../../../RNA_Seq/RNASeqLibrary.tsv

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	../NCBI_output/
	./

no changes added to commit (use "git add" and/or "git commit -a")


In [31]:
! git add $git_experiment_path $git_library_path

In [32]:
! git commit -m $commit_message_exp

[develop b352616] adding annotated bulk experiment SRP065487
 2 files changed, 5 insertions(+)


In [33]:
! git push

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 1.95 KiB | 1.95 MiB/s, done.
Total 5 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
remote: 
remote: To create a merge request for develop, visit:
remote:   https://gitlab.sib.swiss/Bgee/expression-annotations/-/merge_requests/new?merge_request%5Bsource_branch%5D=develop
remote: 
To https://gitlab.sib.swiss/Bgee/expression-annotations.git/
   262b4b4..b352616  develop -> develop


### add annotation folder and script to git

In [34]:
! git pull

Already up to date.


1. run first two cells (annotation summary)
2. export as html

In [ ]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push